# 01-01 — LR Diagnostics (convergence, threshold, C-sweep)

Standalone diagnostic next to ``01-LogisticAnalysis-Aggregated.ipynb``.

Logistic regression is convex — there is no epoch-wise loss curve. The
analogues produced here:

- **Convergence**: train/val log-loss vs optimizer iterations
  (warm-started ``lbfgs`` with increasing ``max_iter`` budgets)
- **Threshold sweep**: validation Balanced Accuracy vs decision threshold
- **C-sweep**: cross-validated Balanced Accuracy vs L2 strength ``C``,
  using the ``lr_C_cv_results_*.csv`` files saved by the main notebook

Inputs: ``results_lr_aggregated/lr_C_cv_results_{drug}.csv`` (from the main
01 run). If absent, a mid-range C is used and a warning is printed.

Outputs (in ``results_lr_diagnostics/``):

- ``01_lr_convergence.pdf`` + ``lr_convergence.csv``
- ``01_lr_threshold_sweep.pdf`` + ``lr_threshold_sweep.csv``
- ``01_lr_C_sweep.pdf`` + ``lr_C_sweep.csv``

Run on Colab. Set env ``SMOKE=1`` for a fast local dry run.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    print('Running locally')
    IN_COLAB = False

if IN_COLAB:
    !pip install maldideepkit maldiamrkit --quiet

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import log_loss, balanced_accuracy_score, roc_auc_score

from maldideepkit.base.data import fit_input_transform, apply_input_transform
from maldiamrkit.evaluation import stratified_species_drug_split

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

SMOKE = os.environ.get("SMOKE", "0") == "1"
SEED = 42
np.random.seed(SEED)

if IN_COLAB:
    DATA_ROOT = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet/Processed")
else:
    DATA_ROOT = Path("/media/asd/home/asd/Projects/Flower/DataSet/Dryad-DataSet/Processed")

OUT_DIR = Path("./results_lr_diagnostics")
OUT_DIR.mkdir(exist_ok=True)
LR_DIR = Path("./results_lr_aggregated")          # main 01 outputs (cv_results + summary)

SITES = ["Proc_DRIAMS-A", "Proc_DRIAMS-B", "Proc_DRIAMS-C", "Proc_DRIAMS-D"]
BIN_COLS = [f"bin_{i}" for i in range(6000)]
DRUGS = ["Ciprofloxacin", "Amoxicillin-Clavulanic_acid", "Gentamicin"]
THRESHOLDS = np.linspace(0.05, 0.95, 91)

STEPS = [5, 10, 25, 50, 100, 250, 500, 1000]
if SMOKE:
    DRUGS = DRUGS[:1]
    STEPS = [5, 10, 25]

print(f"SMOKE={SMOKE} | drugs={DRUGS} | steps={STEPS}")
print(f"Data: {DATA_ROOT}")
print(f"Main-notebook outputs: {LR_DIR.resolve()}")

In [ ]:
# ── Same data loading / splits / preprocessing as 01 ──
def load_aggregated_drug(drug_name):
    frames = []
    for site in SITES:
        p = DATA_ROOT / site / drug_name / "data.csv"
        if p.exists():
            frames.append(pd.read_csv(p))
    if not frames:
        raise FileNotFoundError(f"No data for {drug_name} under {DATA_ROOT}")
    df_all = pd.concat(frames, ignore_index=True)
    X = df_all[BIN_COLS].to_numpy(dtype="float32")
    y = df_all["label"].to_numpy(dtype=int)
    species = df_all["species"].values
    return X, y, species


def create_splits(X, y, species, train_size=0.70, test_size=0.15, seed=SEED):
    n = len(y)
    idx = np.arange(n)
    idx_trval, idx_test, _, _ = stratified_species_drug_split(
        idx.reshape(-1, 1), y, species=species, test_size=test_size, random_state=seed)
    idx_trval = idx_trval.flatten().astype(int)
    idx_test = idx_test.flatten().astype(int)
    X_trval, X_test = X[idx_trval], X[idx_test]
    y_trval, y_test = y[idx_trval], y[idx_test]
    sp_trval = species[idx_trval]
    val_frac = 0.15 / (1 - test_size)
    idx2 = np.arange(len(y_trval))
    idx_train, idx_val, _, _ = stratified_species_drug_split(
        idx2.reshape(-1, 1), y_trval, species=sp_trval, test_size=val_frac, random_state=seed)
    idx_train = idx_train.flatten().astype(int)
    idx_val = idx_val.flatten().astype(int)
    return {"train": (X_trval[idx_train], y_trval[idx_train]),
            "val":   (X_trval[idx_val],   y_trval[idx_val]),
            "test":  (X_test, y_test)}


data = {}
for drug in DRUGS:
    X, y, species = load_aggregated_drug(drug)
    if SMOKE:
        rs = np.random.RandomState(SEED)
        idx = rs.choice(len(y), size=min(1500, len(y)), replace=False)
        X, y, species = X[idx], y[idx], species[idx]
    splits = create_splits(X, y, species)
    state = fit_input_transform(splits["train"][0], "log1p+standardize")
    pp = {part: (apply_input_transform(splits[part][0], state), splits[part][1])
          for part in ("train", "val", "test")}
    data[drug] = pp
    print(f"  {drug:35s} train={pp['train'][0].shape[0]} val={pp['val'][0].shape[0]} test={pp['test'][0].shape[0]}")

In [ ]:
# ── Best C per drug from the main notebook's saved cv_results ──
def best_C_from_cv(drug):
    tag = drug.replace("/", "_").replace(" ", "_").lower()
    p = LR_DIR / f"lr_C_cv_results_{tag}.csv"
    if p.exists():
        df = pd.read_csv(p)
        if "mean_test_score" in df.columns and "param_C" in df.columns:
            return float(df.loc[df["mean_test_score"].idxmax(), "param_C"]), p
    return None, p

BEST_C = {}
for drug in DRUGS:
    C, p = best_C_from_cv(drug)
    if C is None:
        C = 2.5e-4
        print(f"  {drug}: cv_results not found ({p.name}) — falling back to C={C:.1e}")
    BEST_C[drug] = C
    print(f"  {drug:35s} best C = {C:.2e}")

In [ ]:
# ── Convergence: train/val log-loss vs lbfgs iterations (warm start) ──
conv_rows = []
for drug in DRUGS:
    X_train, y_train = data[drug]["train"]
    X_val, y_val = data[drug]["val"]
    C = BEST_C[drug]

    clf = LogisticRegression(penalty="l2", solver="lbfgs", class_weight="balanced",
                             C=C, max_iter=STEPS[0], warm_start=True, random_state=SEED)
    cum_iter = 0
    for budget in STEPS:
        clf.set_params(max_iter=budget)
        clf.fit(X_train, y_train)
        cum_iter += int(np.ravel(clf.n_iter_)[0])
        tr = clf.predict_proba(X_train)[:, 1]
        va = clf.predict_proba(X_val)[:, 1]
        conv_rows.append({
            "drug": drug, "max_iter_budget": budget, "cum_iter": cum_iter,
            "train_logloss": log_loss(y_train, tr, labels=[0, 1]),
            "val_logloss": log_loss(y_val, va, labels=[0, 1]),
            "val_balacc_at_0.5": balanced_accuracy_score(y_val, va >= 0.5),
        })
        print(f"  {drug:35s} budget={budget:5d} cum_iter={cum_iter:6d}  "
              f"train_ll={conv_rows[-1]['train_logloss']:.4f}  val_ll={conv_rows[-1]['val_logloss']:.4f}")

conv_df = pd.DataFrame(conv_rows)
conv_df.to_csv(OUT_DIR / "lr_convergence.csv", index=False)
conv_df

In [ ]:
# ── Threshold sweep on the validation split (converged model at best C) ──
thr_rows = []
for drug in DRUGS:
    X_train, y_train = data[drug]["train"]
    X_val, y_val = data[drug]["val"]
    C = BEST_C[drug]
    clf = LogisticRegression(penalty="l2", solver="lbfgs", class_weight="balanced",
                             C=C, max_iter=5000, random_state=SEED).fit(X_train, y_train)
    va = clf.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, va)
    bal_default = balanced_accuracy_score(y_val, va >= 0.5)
    for t in THRESHOLDS:
        thr_rows.append({"drug": drug, "threshold": t, "val_auc": auc,
                         "val_balacc": balanced_accuracy_score(y_val, va >= t)})
    best = max(thr_rows[-len(THRESHOLDS):], key=lambda r: r["val_balacc"])
    print(f"  {drug:35s} default(0.5) BalAcc={bal_default:.4f}  "
          f"best t={best['threshold']:.3f} BalAcc={best['val_balacc']:.4f}  AUC={auc:.4f}")

thr_df = pd.DataFrame(thr_rows)
thr_df.to_csv(OUT_DIR / "lr_threshold_sweep.csv", index=False)

In [ ]:
# ── C-sweep from the main notebook's saved cv_results ──
cv_frames = []
for drug in DRUGS:
    tag = drug.replace("/", "_").replace(" ", "_").lower()
    p = LR_DIR / f"lr_C_cv_results_{tag}.csv"
    if p.exists():
        d = pd.read_csv(p)
        d["drug"] = drug
        cv_frames.append(d)

if not cv_frames:
    print("No cv_results files found — run the main 01 notebook first (or check LR_DIR).")
else:
    cv_df = pd.concat(cv_frames, ignore_index=True)
    cv_df.to_csv(OUT_DIR / "lr_C_sweep.csv", index=False)

    fig, ax = plt.subplots(figsize=(8, 5))
    for drug in cv_df["drug"].unique():
        d = cv_df[cv_df["drug"] == drug].sort_values("param_C")
        ax.plot(d["param_C"], d["mean_test_score"], marker="o", ms=3, lw=1.8,
                label=f"{drug} (best C={d.loc[d['mean_test_score'].idxmax(), 'param_C']:.1e})")
        if "std_test_score" in d.columns:
            ax.fill_between(d["param_C"], d["mean_test_score"] - d["std_test_score"],
                            d["mean_test_score"] + d["std_test_score"], alpha=0.15)
    ax.set_xscale("log")
    ax.set_xlabel("C (inverse L2 strength)"); ax.set_ylabel("CV Balanced Accuracy")
    ax.set_title("01 — L2 regularization sweep (3-fold CV on train)")
    ax.legend(fontsize=8); ax.grid(True, ls="--", alpha=0.4)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "01_lr_C_sweep.pdf", bbox_inches="tight")
    plt.show()

In [ ]:
# ── Convergence + threshold figures ──
fig, ax = plt.subplots(figsize=(8, 5))
for drug in DRUGS:
    d = conv_df[conv_df["drug"] == drug]
    ax.plot(d["cum_iter"], d["train_logloss"], lw=1.8, label=f"{drug} train")
    ax.plot(d["cum_iter"], d["val_logloss"], lw=1.8, ls="--", label=f"{drug} val")
ax.set_xlabel("Cumulative lbfgs iterations"); ax.set_ylabel("log-loss")
ax.set_title("01 — L2 LR convergence (convex objective)")
ax.legend(fontsize=8); ax.grid(True, ls="--", alpha=0.4)
plt.tight_layout()
plt.savefig(OUT_DIR / "01_lr_convergence.pdf", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
for drug in DRUGS:
    d = thr_df[thr_df["drug"] == drug]
    ax.plot(d["threshold"], d["val_balacc"], lw=1.8, label=drug)
    bt = d.loc[d["val_balacc"].idxmax()]
    ax.axvline(bt["threshold"], ls=":", lw=1, alpha=0.6)
ax.axvline(0.5, color="gray", ls="--", lw=1, label="default 0.5")
ax.set_xlabel("Decision threshold"); ax.set_ylabel("Validation Balanced Accuracy")
ax.set_title("01 — Threshold sweep")
ax.legend(fontsize=8); ax.grid(True, ls="--", alpha=0.4)
plt.tight_layout()
plt.savefig(OUT_DIR / "01_lr_threshold_sweep.pdf", bbox_inches="tight")
plt.show()

## What to look for

- **Convergence**: LR is convex; both train and val log-loss should decrease
  monotonically and flatten. A gap that stays constant is the price of L2 /
  class weighting, not overfitting in the epoch sense.
- **Threshold sweep**: if the optimum sits away from 0.5, threshold tuning is
  doing real work (the usual case for imbalanced resistance labels).
- **C-sweep**: the CV curve shape shows whether the L2 search range is
  interior (optimum not at an edge) — edge optima mean the grid should move.
- These curves are the LR analogues of the MLP loss curves in ``02-01``;
  they are the evidence behind 01's Key Findings.